In [1]:
from config2 import *

from multivae.models import AutoModel
from multivae.trainers import AddDccaTrainer, AddDccaTrainerConfig
from multivae.models.nn.default_architectures import MultipleHeadJointEncoder
from multivae.models.nn.mmnist import DecoderConvMMNIST, EncoderConvMMNIST_adapted
from torch.utils.data import DataLoader
from multivae.data.utils import set_inputs_to_device
from tqdm import tqdm



train_data = MMNISTDataset(
    data_path="/Users/agathe/dev/data",
    split="train",
    missing_ratio=0,
    keep_incomplete=True,
)

test_data = MMNISTDataset(data_path="/Users/agathe/dev/data", split="test")

train_data, eval_data = random_split(
    train_data, [0.9, 0.1], generator=torch.Generator().manual_seed(0)
)

device = 'cpu'

model = AutoModel.load_from_folder('/Users/agathe/dev/multivae_package/JNFDcca_training_2023-07-11_13-24-45/final_model')
model = model.to(device)

#### Compute all embeddings for the DCCA ####

test_loader = DataLoader(test_data,10)

batch  = next(iter(test_loader))
print(test_data[0].data['m0'])



model.dcca_networks['m0']


/Users/agathe/dev/multivae_package/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[[0.4471, 0.2353, 0.0510,  ..., 0.9490, 0.9725, 0.9569],
         [0.5176, 0.2353, 0.1333,  ..., 0.8980, 0.9294, 0.9373],
         [0.5255, 0.3098, 0.3804,  ..., 0.8706, 0.9098, 0.9333],
         ...,
         [0.2431, 0.2706, 0.2471,  ..., 0.8588, 0.4824, 0.2314],
         [0.4157, 0.5490, 0.5765,  ..., 0.8078, 0.5961, 0.4000],
         [0.6000, 0.8235, 0.9412,  ..., 0.7059, 0.6549, 0.5922]],

        [[0.5137, 0.2157, 0.0392,  ..., 0.9765, 1.0000, 0.9961],
         [0.5882, 0.2235, 0.1294,  ..., 0.9373, 0.9843, 0.9922],
         [0.5961, 0.2980, 0.3765,  ..., 0.9255, 0.9843, 1.0000],
         ...,
         [0.3059, 0.3216, 0.2863,  ..., 0.8980, 0.5255, 0.2588],
         [0.4471, 0.5922, 0.6118,  ..., 0.8471, 0.6353, 0.4353],
         [0.6196, 0.8588, 0.9765,  ..., 0.7451, 0.6941, 0.6275]],

        [[0.5529, 0.2941, 0.1059,  ..., 1.0000, 1.0000, 1.0000],
         [0.6275, 0.2902, 0.1922,  ..., 0.9765, 1.0000, 1.0000],
         [0.6431, 0.3647, 0.4392,  ..., 0.9608, 1.0000, 1.

EncoderConvMMNIST_adapted(
  (shared_encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
  )
  (class_mu): Conv2d(128, 32, kernel_size=(4, 4), stride=(2, 2))
  (class_logvar): Conv2d(128, 32, kernel_size=(4, 4), stride=(2, 2))
)

In [ ]:

embeddings = {k : [] for k in model.dcca_networks}
labels = []
for batch in tqdm(test_loader):
    
    batch = set_inputs_to_device(batch , device=device)
    
    for m in batch.data:
        enc = model.dcca_networks[m](batch.data[m])
        embeddings[m].append(enc)
        
    labels.append(batch.labels)
        

embeddings = {k : torch.cat(embeddings[k], dim=0) for k in embeddings}
labels = torch.cat(labels, dim=0)
